# Downloading HydroLakes (lake polygons, v10) from Earth Engine

Source: `projects/sat-io/open-datasets/HydroLakes/lake_poly_v10` (HydroSHEDS / HydroLakes v1.0).

Each polygon has a `Lake_type` attribute:
- `1` — Lake (natural)
- `2` — Reservoir (artificial)
- `3` — Lake control (natural lake with regulated outflow, e.g. hydropower)

All three types are kept so you can disaggregate water-loss measures by lake type downstream.

Exports go to Google Drive via Earth Engine. Monitor progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import os
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize()

In [3]:
# Global ROI (matches the other waterchange exports)
world_bbox = ee.Geometry.BBox(-180, -85, 180, 85)

# Load HydroLakes v10 (all three Lake_type categories preserved)
lakes = ee.FeatureCollection('projects/sat-io/open-datasets/HydroLakes/lake_poly_v10')

# Restrict to the ROI (global bbox here — drop or replace with a smaller geometry if you want a regional subset)
lakes_in_roi = lakes.filterBounds(world_bbox)

print('Total features in ROI:', lakes_in_roi.size().getInfo())

# Shapefiles require a single geometry type per file. The sat-io HydroLakes
# collection contains a handful of LineString features that would otherwise
# break the export with:
#   "Shapefiles cannot contain multiple geometry types; found 'LineString', 'Polygon'"
# Filter to polygon-like geometries only (Polygon + MultiPolygon both write
# fine to a shapefile's Polygon type).
lakes_polys = (
    lakes_in_roi
    .map(lambda f: f.set('geom_type', f.geometry().type()))
    .filter(ee.Filter.inList('geom_type', ['Polygon', 'MultiPolygon']))
)

print('Polygon-only features:', lakes_polys.size().getInfo())

Total features in ROI: 1427688
Polygon-only features: 1427645


In [4]:
# Quick sanity check: feature count per Lake_type (1=lake, 2=reservoir, 3=lake control),
# after filtering to polygon-only features.
for t in [1, 2, 3]:
    n = lakes_polys.filter(ee.Filter.eq('Lake_type', t)).size().getInfo()
    print(f'Lake_type={t}: {n} features')

Lake_type=1: 1420850 features
Lake_type=2: 6686 features
Lake_type=3: 110 features


In [5]:
# --- Export all HydroLakes polygons (Lake_type kept as attribute) ---
# Single shapefile export covering all 3 Lake_type categories with EVERY
# HydroLakes attribute preserved (selectors=None). Uses `lakes_polys` to
# avoid the mixed-geometry shapefile error.
task = ee.batch.Export.table.toDrive(
    collection=lakes_polys,
    description='hydrolakes_v10_all',
    folder='GEE_exports',
    fileNamePrefix='hydrolakes_v10_all',
    fileFormat='SHP',
    selectors=None
)
task.start()

In [6]:
# --- Optional: separate shapefile per Lake_type ---
# Commented out — the single `hydrolakes_v10_all` export above already keeps
# Lake_type as an attribute, so filtering by type after loading the SHP in
# Python is enough. Uncomment this block only if you want three pre-split
# shapefiles (one per Lake_type).
#
# type_labels = {1: 'lake', 2: 'reservoir', 3: 'lakecontrol'}
#
# for t, label in type_labels.items():
#     subset = lakes_polys.filter(ee.Filter.eq('Lake_type', t))
#     task = ee.batch.Export.table.toDrive(
#         collection=subset,
#         description=f'hydrolakes_v10_{label}',
#         folder='GEE_exports',
#         fileNamePrefix=f'hydrolakes_v10_{label}',
#         fileFormat='SHP',
#         selectors=None
#     )
#     task.start()
#     print(f'Started export: hydrolakes_v10_{label}')

### NOTES
- Track export progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks). HydroLakes v10 has ~1.4M features globally, so a single global SHP export can take a while.
- After downloads finish, place the shapefile components (`.shp`, `.shx`, `.dbf`, `.prj`) under `Measures_work/maps/raw/HydroLakes/` so a downstream `*_ethnologue.ipynb` can read them with `gpd.read_file(...)`.
- If you only need a single category (e.g. reservoirs), it's faster to keep just the `hl-export-by-type` cell and comment out the all-types export.